# Chapter 5 - Lab 1: <font color='blue'>Building the Deep Search Planner</font>

**<font color='purple'>Goal</font>**:
In this lab you will build the **planner** — the first stage of the Deep Search Agent. Given a research question, the planner produces a *structured research plan*: a list of sub-tasks with explicit dependencies, ready to be executed.

You will see how **PydanticAI's structured output** removes a whole class of problems: instead of asking an LLM for JSON and praying, you declare a Pydantic schema and the framework guarantees the model returns exactly that shape.

By the end of the lab you will have a planner that turns a question like *"Compare NVIDIA, AMD, and Intel on growth and margins"* into a runnable DAG of 5-10 sub-tasks.

**<font color='purple'>Tech stack</font>**:

* **PydanticAI** (`pydantic-ai`) — structured-output agents with model-string provider config.
* **OpenAI** `gpt-4o` — the planner needs strong reasoning to decompose a question well.
* **Pydantic v2** — typed schemas for `SubTask` and `ResearchPlan`.

You will need an OpenAI API key.

## 1. Install packages

PydanticAI ships with OpenAI support; we add `python-dotenv` for local key loading.

In [ ]:
%pip install -q pydantic-ai pydantic python-dotenv

## 2. Set up the OpenAI API key

If you are in **Google Colab**, add your OpenAI key in the left vertical menu under the name `OPENAI_API_KEY`. Locally, replace the cell below with a `.env` load or a direct assignment.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally — assume the env var is already set.
    pass

## 3. Bootstrap the shared models and helpers

Chapter 5 labs share a `common.py` module that defines the Pydantic models and helpers used across the chapter. If you have cloned the book's repository, `common.py` is already on disk; otherwise the cell below downloads it for you.

In [ ]:
import os, urllib.request

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%205/common.py',
        'common.py',
    )

from common import (
    ResearchPlan, SubTask, TaskStatus,
    PLANNING_MODEL, format_plan,
)

print('Shared models loaded.')

## 4. The planner system prompt

The planner is mostly defined by its system prompt. The prompt explains what data sources are available, what makes a good plan, and what rules the LLM must follow. Spend time reading this — it is the heart of the planner.

In [ ]:
PLANNER_SYSTEM_PROMPT = """\
You are a financial research planner. Given a research question, decompose it
into concrete sub-tasks that can be executed independently by tools.

Available data sources for each sub-task:
- financial_api: Structured financial metrics (revenue, EPS, margins, P/E,
  market cap, price). Best for quantitative data.
- sec_filing: SEC filing content (risk factors, business descriptions, MD&A
  sections). Best for qualitative company-specific information.
- web_search: Recent news, competitive intelligence, analyst commentary.
  Best for current events and market context.

Rules for creating the plan:
1. Each sub-task must have a clear, measurable outcome.
2. Specify which data_sources each sub-task should use (can be multiple).
3. Mark dependencies — which task IDs must complete before this task can start.
4. Tasks with no dependencies can execute in parallel — maximize parallelism.
5. For multi-company comparisons, create parallel tasks for each company.
6. Include a comparative analysis task that depends on all company data tasks.
7. The final task should synthesize all findings — it depends on everything.
8. Use sequential IDs starting from 1.
9. Aim for 5-10 sub-tasks. Too few = insufficient depth. Too many = overhead.
10. If the question is about a single company, still gather peer context.\
"""

## 5. The planner agent (with structured output)

Here is the critical idea: the agent's `output_type` is `ResearchPlan`. PydanticAI takes care of the JSON schema generation, the validation, and the retries — you just declare the type and ask for it. If the model returns something invalid, PydanticAI re-prompts up to `retries` times.

In [ ]:
from pydantic_ai import Agent

planner_agent = Agent(
    model=PLANNING_MODEL,
    system_prompt=PLANNER_SYSTEM_PROMPT,
    output_type=ResearchPlan,
    retries=2,
)

## 6. Generating a plan

We wrap the agent call in `create_research_plan` so callers can pass extra context (used during replanning when validation fails). Try it on a real-world question — compare three semiconductor companies on growth and margins.

In [ ]:
async def create_research_plan(
    question: str, additional_context: str = '',
) -> ResearchPlan:
    user_message = question
    if additional_context:
        user_message += f'\n\nAdditional context:\n{additional_context}'
    result = await planner_agent.run(user_message)
    return result.output

In [ ]:
question = (
    "Analyze NVIDIA's financial performance and competitive position in the "
    "AI chip market. Compare with AMD and Intel on revenue growth, "
    "profitability, and key risk factors."
)

plan = await create_research_plan(question)
print(format_plan(plan))

## 7. Validating the plan

PydanticAI guarantees the *shape* of the plan is correct — every field present, every type right. But a structurally valid plan can still be *semantically* broken: circular dependencies, deadlocks (every task depends on something), or impossibly few sub-tasks. We add a small deterministic validator to catch these.

In [ ]:
def validate_plan(plan: ResearchPlan) -> None:
    task_ids = {t.id for t in plan.sub_tasks}

    if len(plan.sub_tasks) < 2:
        raise ValueError(
            f'Plan has only {len(plan.sub_tasks)} sub-tasks. '
            f'Need at least 2.'
        )

    for task in plan.sub_tasks:
        for dep in task.dependencies:
            if dep not in task_ids:
                raise ValueError(f'Task {task.id} depends on missing task {dep}')
            if dep == task.id:
                raise ValueError(f'Task {task.id} depends on itself')

    startable = [t for t in plan.sub_tasks if not t.dependencies]
    if not startable:
        raise ValueError('No tasks can start (deadlock)')

In [ ]:
validate_plan(plan)
print('Plan validates cleanly.')

## 8. Inspecting parallelism

A good plan maximises parallelism — tasks with no dependencies can run concurrently in a single wave. Plot the dependency structure so you can see what the planner produced.

In [ ]:
parallel = [t for t in plan.sub_tasks if not t.dependencies]
sequential = [t for t in plan.sub_tasks if t.dependencies]

print(f'Parallel-ready tasks: {len(parallel)}')
for t in parallel:
    print(f'  [{t.id}] {t.description[:80]}')

print(f'\nSequential tasks: {len(sequential)}')
for t in sequential:
    print(f'  [{t.id}] depends on {t.dependencies}: {t.description[:60]}')

## 9. Results

You should see a plan with roughly 5-10 sub-tasks: one parallel task per ticker (NVDA, AMD, INTC) for data gathering, plus a comparative analysis task that depends on the three data tasks, and a final synthesis task that depends on everything.

**What to notice about the planner:**

* **Structured output is transformative.** With `output_type=ResearchPlan` you never write parsing code — the model returns a typed object.
* **Validation has two layers**: PydanticAI guarantees the schema; your `validate_plan` guarantees the *semantics* (no cycles, no deadlocks, enough tasks).
* **The system prompt is the planner.** Most of the planner's quality comes from how you describe the data sources and the rules — not from the model choice.
* In Lab 2 you will plug this planner into the full Deep Search pipeline (execute → validate → synthesize) and see how the agent behaves end-to-end.